# Episodic Memory with Write-Time Reflection

This notebook demonstrates the `reflective_memory()` factory, which gives an
`LLMAgent` episodic memory that distils a short lesson from each completed task at
write time.

The example is drawn directly from the
[Reflexion paper (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366), which
frames the pattern as verbal reinforcement learning: a self-reflection model converts
a completed episode into a natural-language lesson, and episodic memory stores that
lesson for future retrieval.

The task is Monte Carlo estimation of pi — one round, two tool calls. The agent calls
`generate_random_sample(n)` to seed a sample and `monte_carlo_estimate` to get the
estimate; the only decision is choosing `n`. A custom `template` supplies the true
value of pi, giving the reflection model a concrete error signal: it can reason about
how far off the estimate was and recommend a specific sample size for next time.

Two properties are illustrated:

- **Write-time reflection.** After the task completes, `reflective_memory()` calls
  the LLM to produce a one-sentence lesson from the task and its outcome. That lesson
  is attached to the episode under `metadata["reflection"]` and rendered automatically
  inside a `<reflection>` tag whenever the episode is displayed.
- **Recalled lessons.** When a new task arrives, `recall()` surfaces the most
  semantically similar episode. Because the episode carries a pre-distilled lesson,
  the agent can apply the insight on the next attempt rather than rediscovering the
  same mistakes.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [14]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

✓ Ollama already running at http://localhost:11434


In [ ]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory.recipes import reflective_memory

## Part 1 — Tools for Monte Carlo Pi Estimation

The tools here are adapted from Capstone 1, which applies the same Monte Carlo
algorithm in a more open-ended multi-step task. Two of the three capstone tools are
wired to the agent in this demo:

- `generate_random_sample(n)` — generates `n` random 2-D points in [0, 1] × [0, 1],
  stores them in a global registry, and returns a `sample_id`.
- `monte_carlo_estimate(sample_id)` — counts the fraction of stored points that fall
  inside the unit circle, multiplies by 4, and returns the estimate alongside the
  sample size.

Capstone 1 also defines a third tool, `add_more_points_to_sample`, which lets an
agent grow an existing sample incrementally without discarding accumulated data. It
is omitted here because the task constrains the agent to one round — a single
`generate_random_sample` call followed by a single `monte_carlo_estimate`. The only
decision is choosing `n`.

In [16]:
import uuid

import numpy as np
from pydantic import BaseModel, ConfigDict, Field, computed_field

from llm_agents_from_scratch.tools import PydanticFunctionTool

SAMPLE_REGISTRY: dict[str, list[tuple[float, float]]] = {}


class RandomSampleParams(BaseModel):
    """Params for generate_random_sample."""

    model_config = ConfigDict(extra="forbid")
    n: int = Field(description="Number of random points to generate.")


class RandomSample(BaseModel):
    """Result from generate_random_sample."""

    sample_id: str = Field(
        description="Pass this sample_id to monte_carlo_estimate.",
    )

    @computed_field
    @property
    def sample_size(self) -> int:
        """Current number of points in the sample."""
        return len(SAMPLE_REGISTRY[self.sample_id])

    def __str__(self) -> str:
        """JSON string representation."""
        return self.model_dump_json()


def generate_random_sample(params: RandomSampleParams) -> RandomSample:
    """Generate n random points in [0,1]x[0,1] and store them.

    Returns a sample_id. Pass this sample_id to monte_carlo_estimate.
    """
    pts = np.random.uniform(size=(params.n, 2))
    sample_id = str(uuid.uuid4())
    SAMPLE_REGISTRY[sample_id] = [tuple(pt) for pt in pts.tolist()]
    return RandomSample(sample_id=sample_id)


class MonteCarloParams(BaseModel):
    """Params for monte_carlo_estimate."""

    model_config = ConfigDict(extra="forbid")
    sample_id: str = Field(
        description="The sample_id returned by generate_random_sample.",
    )


class MonteCarloResult(BaseModel):
    """Result from monte_carlo_estimate."""

    sample_id: str
    sample_size: int
    estimate: float

    def __str__(self) -> str:
        """JSON string representation."""
        return self.model_dump_json()


def monte_carlo_estimate(params: MonteCarloParams) -> MonteCarloResult:
    """Estimate pi using the Monte Carlo method.

    Counts the fraction of stored points that fall inside the unit circle
    and multiplies by 4.
    """
    points = SAMPLE_REGISTRY[params.sample_id]
    n = len(points)
    inside = sum(x**2 + y**2 < 1 for x, y in points)
    return MonteCarloResult(
        sample_id=params.sample_id,
        sample_size=n,
        estimate=(inside / n) * 4,
    )


random_sample_tool = PydanticFunctionTool(generate_random_sample)
mc_estimate_tool = PydanticFunctionTool(monte_carlo_estimate)

## Creating the Agent with Memory

`reflective_memory()` creates a `Memory` backed by an in-process Qdrant vector store.
Before each episode is written it calls `llm.complete()` to distil a one-sentence
lesson from the task and result, storing it under `metadata["reflection"]`.

The optional `template` parameter replaces the built-in reflection prompt. For this
demo it is replaced with a task-specific prompt that supplies the true value of pi.
This gives the reflection model an error signal: it can compute how far off the
estimate was and recommend a concrete sample size for the next attempt — rather than
giving generic advice about "using large samples".

For tasks where correctness is not obvious from the result alone, pass a custom
`template` that supplies a ground truth or evaluation rubric. The default template
works well when success and failure are unambiguous; a custom template is the hook
for turning a generic observation into a precise, actionable lesson.

In [ ]:
REFLECTION_TEMPLATE = """\
You just attempted to estimate pi using the Monte Carlo method.

Task: {instruction}
Result: {result}

The true value of pi is 3.141592653589793.

In one sentence, what specific sample size should be used next time to \
achieve a more accurate estimate, and why?\
"""

llm = OllamaLLM(model="qwen3:14b", think=False)
memory = reflective_memory(llm=llm, max_results=1, template=REFLECTION_TEMPLATE)
agent = LLMAgent(
    llm=llm,
    tools=[random_sample_tool, mc_estimate_tool],
    memories=[memory],
)

## Part 2 — Running the Pi Estimation Task

The agent gets one round: call `generate_random_sample(n)` to generate a sample,
then call `monte_carlo_estimate` to get the estimate. The only decision is choosing
`n`.

That single choice is where everything happens. Too small and the estimate is
unreliable; large enough and it converges to the right answer. The ALL CAPS warning
in the instruction is a hint — the reflection will capture whether the agent heeded
it.

In [18]:
TASK_INSTRUCTION = (
    "Use the Monte Carlo method to estimate pi in one round.\n\n"
    "IMPORTANT: MONTE CARLO ESTIMATES OF PI CONVERGE VERY SLOWLY. "
    "YOU WILL NEED MILLIONS OF POINTS FOR A RELIABLE RESULT.\n\n"
    "Steps:\n"
    "1. Call generate_random_sample(n) — choose n to maximise accuracy.\n"
    "2. Call monte_carlo_estimate(sample_id) — this is your final estimate.\n\n"
    "You have two tools:\n"
    "- generate_random_sample(n): generates n random 2-D points in "
    "[0,1]x[0,1] and returns a sample_id.\n"
    "- monte_carlo_estimate(sample_id): computes a pi estimate from the "
    "stored sample.\n\n"
    "When done, respond with exactly:\n"
    '{"sample_id": "<the-sample-id>"}'
)

In [19]:
enable_console_logging(logging.INFO)

task1 = Task(instruction=TASK_INSTRUCTION)
result1 = await agent.run(task1)
print(result1)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Use the Monte Carlo method to estimate pi in one round.

IMPORTANT: MONTE CARLO ESTIMATES OF PI CONVERGE VERY SLOWLY. YOU WILL NEED M...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Use the Monte Carlo method to estimate pi in one round.

IMPORTANT: MONTE CARLO ESTIMATES OF PI CONVERGE VERY SLOWLY. YOU WILL NEE...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to call the generate_random_sample tool with a sufficiently large n to maximize accuracy. Let's choose n = 1000000.
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call generate_random_sample(1000000)
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Call generate_random_sample(1000000)
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: generate_random_sample


INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: {"sample_id":"baf2c906-6327-4bbb-98c6-146fa166b037","sample_size":1000000}
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I have generated a random sample with sample_id "baf2c906-6327-4bbb-98c6-146fa166b037" and sample_size 1000000. Now, I need to call the...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call monte_carlo_estimate with sample_id 'baf2c906-6327-4bbb-98c6-146fa166b037' to compute the pi estimate.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Call monte_carlo_estimate with sample_id 'baf2c906-6327-4bbb-98c6-146fa166b037' to compute the pi estimate.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: monte_carlo_estimate
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: {"sample_id":"baf2c906-6327-4bbb-98c6-146fa166b037","sample_size":1000000,"estimate":3.145016}
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: The Monte Carlo estimate of p

In [20]:
print(await memory.summary())

ReflectiveMemory (recall last 1):
JSONMemoryStore: 1 episodes | path=reflective_episodes.jsonl
  newest: 2026-05-23 01:00:23 | Use the Monte Carlo method to estimate pi in one round.

IMP
  oldest: 2026-05-23 01:00:23 | Use the Monte Carlo method to estimate pi in one round.

IMP


## The Recency Baseline

Before looking at the reflection, consider what `recency_memory()` would surface for
a new task. A recency-based memory stores no additional metadata. The recalled
episode carries the full result — which tools were called, what estimate was returned
— but the lesson is implicit.

A future agent seeing this raw episode would need to reason from the estimate alone
to figure out what went wrong and by how much. The signal is there, but buried.

In [ ]:
print("What recency_memory() would recall (no reflection):\n")
recent_eps = await memory.store.read_recent(memory.store.max_results)
for ep in recent_eps:
    ep_without_reflection = Episode(
        task=ep.task,
        rollout=ep.rollout,
        result=ep.result,
        completed_at=ep.completed_at,
    )
    print(str(ep_without_reflection))
    print()

## Part 3 — Inspecting the Reflection

Now look at the same episode as stored by `reflective_memory()`. It carries a
`<reflection>` tag with the one-sentence lesson distilled at write time.

Because the custom `template` supplies the true value of pi, the reflection model
has a concrete error signal: it can compute how far off the estimate was and
recommend a specific sample size for next time. The lesson is captured in a single
sentence rather than buried in the full trajectory.

In [ ]:
print("What reflective_memory() recalls (with reflection):\n")
recent_eps = await memory.store.read_recent(memory.store.max_results)
for ep in recent_eps:
    print(str(ep))
    print()

## Part 4 — A Recalled Lesson Guides the Next Run

The same task is submitted a second time. Before the agent takes its first step,
`recall()` injects the episode from Part 2 — with its embedded reflection — into the
system prompt.

If Part 2 produced an inaccurate estimate, the reflection names exactly what to do
differently. If Part 2 succeeded, the reflection confirms the winning strategy.
Either way the agent starts this run with a concrete lesson rather than a blank
slate.

Check the logs: does the agent apply the recalled insight on its first tool call?

In [ ]:
print("Episode recalled for the second task:\n")
recalled = await memory.store.read_recent(memory.store.max_results)
for ep in recalled:
    print(str(ep))
    print()

In [24]:
enable_console_logging(logging.INFO)

task4 = Task(instruction=TASK_INSTRUCTION)
result4 = await agent.run(task4)
print(result4)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Use the Monte Carlo method to estimate pi in one round.

IMPORTANT: MONTE CARLO ESTIMATES OF PI CONVERGE VERY SLOWLY. YOU WILL NEED M...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Use the Monte Carlo method to estimate pi in one round.

IMPORTANT: MONTE CARLO ESTIMATES OF PI CONVERGE VERY SLOWLY. YOU WILL NEE...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: generate_random_sample
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: {"sample_id":"95513b79-e71c-4254-8ce9-78a826c20775","sample_size":10000000}
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: {"name": "monte_carlo_estimate", "arguments": {"sample_id":"95513b79-e71c-4254-8ce9-78a826c20775"}}
</tool_call>
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: The assistant needs to call the 'monte_carlo_estimate' tool with the provided sample_id to compute the final estimate of pi.
INFO (llm_agents_fs.T

## Key Takeaway

`reflective_memory()` converts task experience into transferable lessons. The
Reflexion paper frames this as verbal reinforcement learning: rather than updating
model weights, the agent stores natural-language feedback from each trial and
retrieves it when facing the same problem.

The Pi estimation task makes the mechanism concrete because the custom `template`
supplies the true value of pi, giving the reflection model an explicit error signal
to reason from. In the run above:

- **Task 1** used 1 million points and produced an estimate of `3.145016` — an error
  of roughly 0.003 from the true value.
- The reflection identified the 1/√n convergence rate and recommended **10 million
  points** by name as the next sample size.
- **Task 2** opened immediately with `generate_random_sample(10000000)`, produced
  `3.1415376`, and reduced the error to roughly 0.00006 — about 60× more accurate.

The comparison in Parts 2 and 3 shows why the custom template matters. The
`recency_memory()` episode carries the same result (`3.145016`) but no lesson — a
future agent re-reading it would only know the estimate was off, not by how much or
what to do about it. The `reflective_memory()` episode distils the corrective lesson
in one sentence: a specific sample size, grounded in the mathematical reason it will
work better.

The factory takes three parameters: `llm` (used for reflection), `max_results` (how
many similar episodes to recall, defaults to `5`), and `template` (optional custom
reflection prompt, defaults to a general-purpose one-sentence lesson template).
Swapping in `recency_memory()` or `similarity_memory()` requires changing only the
one line that constructs the memory object.